# einops-einsum — ex10: batched bilinear with broadcast of non-batched matrix — y_b = u_b^T A u_b + v_b^T A v_b

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. Running the final beacon cell reports progress against the `Einops: Deep Learning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-einsum`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einsum 4-tensor broadcast — quick refresher

`einsum` lets you mix batched and non-batched operands by **omitting** the batch index from the non-batched operand's pattern. Operands without the batch index are broadcast across every batch slot.

```python
# u, v: (B, D)   A, B_mat: (D, D)   broadcast A and B_mat across B
y = einsum('b i, i j, b j, b k, k l, b l -> b', u, A, u, v, B_mat, v)
```

This computes `y_b = u_b^T A u_b + v_b^T B v_b` in one call — except we'd usually split it because a single einsum pattern computes ONE sum, not a SUM OF two terms. (We'll handle that by computing the two bilinear forms separately and adding.)

**This drill (ex10) vs ex1-9.** ex8 did a single bilinear `y_b = x_b^T A x_b` with `A` batched. ex10 generalizes to TWO vectors u, v on the same `A` (now NON-batched, broadcast across the batch axis), summed into one scalar per batch element. Verifies the broadcast-non-batched-axis-into-batched-einsum pattern.

### Exercise 10 — batched bilinear with broadcast of non-batched matrix — y_b = u_b^T A u_b + v_b^T A v_b

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Bloom level: Create
> LO: Create a single bilinear-form computation that mixes batched vectors (`u, v`: shape `(B, D)`) with a non-batched matrix (`A`: shape `(D, D)`), broadcasting `A` across the batch axis via index-omission in the einsum pattern.
> Keywords: bilinear, broadcast-non-batched, multi-axis, einsum
> ```

**KCs targeted:** `einsum-omit-axis-broadcasts`, `einsum-multi-operand-batched`

Implement `ex10_double_bilinear(u, v, A)`.

Compute the batched quadratic-form sum:
  `y_b = u_b^T A u_b + v_b^T A v_b`

where `u, v` have shape `(B, D)` and `A` is a **single** `(D, D)` matrix shared across the whole batch.

Rules:
1. Use exactly **two** `t.einsum` calls — one per bilinear term. Each call must broadcast `A` across the batch by omitting `b` from `A`'s index string.
2. Both calls follow the pattern `einsum('b i, i j, b j -> b', x, A, x)`. Note `A`'s indices are just `'i j'` (no `b`); this is what makes the broadcast happen.
3. Sum the two terms, return a `(B,)` tensor.

Inputs: `u`, `v` shape `(B, D)`; `A` shape `(D, D)`.
Output: `(B,)` tensor `y` where `y[b] = u[b]^T A u[b] + v[b]^T A v[b]`.

The visualization is a heatmap over a 2-D parameter grid: vary `u` along one axis and `v` along another (both scaled multiples of unit vectors), and plot `y_b` over the grid. For positive-definite `A` you should see an elliptic-paraboloid contour.

In [ ]:
def ex10_double_bilinear(u: Tensor, v: Tensor, A: Tensor) -> Tensor:
    y_u = t.einsum('b i, i j, b j -> b', u, A, u)
    y_v = t.einsum('b i, i j, b j -> b', v, A, v)
    return y_u + y_v


<details><summary>Solution</summary>

```python
def ex10_double_bilinear(u: Tensor, v: Tensor, A: Tensor) -> Tensor:
    y_u = t.einsum('b i, i j, b j -> b', u, A, u)
    y_v = t.einsum('b i, i j, b j -> b', v, A, v)
    return y_u + y_v
```

**Why omitting `b` from `A` is the broadcast.** In einsum index notation, an index that appears in some operands but not others is implicitly broadcast over the absent ones. `'b i, i j, b j -> b'` says: `A` lacks `b` → broadcast `A` across the batch; `u` has `b` and `i` → batched over `b`, contracts over `i, j` with `A` and `u`'s second copy. Same as `u_b^T A u_b`.

**Why two einsum calls instead of one giant pattern.** A single einsum computes ONE multi-index sum. To produce `y_b = u_b^T A u_b + v_b^T A v_b` you'd need a sum-of-two-products, which einsum doesn't express as a single term. Two patterns + Python `+` is the right factoring; trying to cram both into one call via auxiliary indices fast becomes unreadable.

**When to reach for this pattern.** Anywhere you have a shared metric tensor that defines an inner product (Riemannian manifolds, learned distance metrics, attention with a shared key/query projection across heads). The non-batched-broadcast saves memory: you keep `A` as `(D, D)` instead of expanding it to `(B, D, D)` just to feed it through a matmul.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()